# Introduction for LangChain for Data Professionals

Below code is from a pluralsigh course.

In [1]:
import os
from dotenv import load_dotenv

In [2]:
os.environ.get('OPENAI_API_KEY', "No key found")

'No key found'

In [3]:
import sys
import os

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key

from openai import OpenAI

**remark** 
what is strange that depending on the import the model seems to memorize the history while the other does not

In [10]:
from langchain.llms import OpenAI
#from langchain_openai import ChatOpenAI
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage

In [11]:
# Define a prompt text
text = "What would be a good name for a new national park with a jungle terrain?"

In [12]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": text}]
)

In [13]:
response.choices[0].message.content

'Emerald Canopy National Park'

In [14]:
# Create a HumanMessage schema object containing the prompt
message = [HumanMessage(content=text)]

In [15]:
# create a langchain chat model
chat_model = ChatOpenAI(model="gpt-4o-mini", api_key=api_key)

In [16]:
chat_model.invoke(message)

AIMessage(content='Here are some suggestions for names for a new national park with a jungle terrain:\n\n1. **Emerald Canopy National Park**\n2. **Rainforest Oasis National Park**\n3. **Verdant Wilds National Park**\n4. **Tropical Haven National Park**\n5. **Lush Jungle Preserve**\n6. **Biodiversity Basin National Park**\n7. **Mystic Jungle National Park**\n8. **Serengeti Woods National Park**\n9. **Wildheart Rainforest National Park**\n10. **Amazonian Echoes National Park**\n\nEach of these names evokes the lush, vibrant nature of a jungle while conveying a sense of preservation and wonder.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 134, 'prompt_tokens': 23, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_finge

In [17]:
prompt = "What would be a good name for a new national park with jungle terrain? Give one name."
response = chat_model.invoke(prompt)
print(response.content)

followup = "could you provide 3 more potential names"
response2 = chat_model.invoke(followup)
print(response2.content)

followup2 = "Could you provide 3 more potential names, which has something related to jungle in the name?"
response3 = chat_model.invoke(followup2)
print(response3.content)


Emerald Canopy National Park
Of course! Could you please provide a bit more context about what type of names you are looking for? For example, are they for a business, a character in a story, a product, or something else?
Sure! Here are three potential names that relate to the jungle:

1. **Jungle Quest**
2. **Tropical Jungle Adventure**
3. **Jungle Oasis** 

Let me know if you need more suggestions!


In [18]:
prompt = "What would be a good name for a new national park with jungle terrain? Give one name."
response = chat_model.invoke(prompt)
print(response.content)

followup = "Could you suggest 5 more names for this jungle park with wild animals?"
alternatives = chat_model.invoke(followup).content
print("\nother options")
for name in alternatives.split("\n"):
    print(f"- {name}")

Emerald Canopy National Park

other options
- Sure! Here are five name suggestions for your jungle park with wild animals:
- 
- 1. **Wild Haven Jungle Park**
- 2. **Savanna Safari Adventure**
- 3. **Untamed Wilderness Park**
- 4. **Jungle Quest Wildlife Sanctuary**
- 5. **Tropical Wilds Exploration Zone**
- 
- Feel free to mix and match or modify any of these suggestions to better fit your vision!


Normal `OpenAI` workflow 

In [ ]:
from langchain.prompts import PromptTemplate

text = "What would be a good name for a new national park with a {terrain} terrain."
messages = [{"role": "user", "content": text.format(terrain="jungle")}]

response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0)
print(response.choices[0].message.content)

`LangChain` workflow 1

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
#from langchain.schema import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key, temperature=0)
terrain = "jungle"

messages = [HumanMessage(content=text.format(terrain="jungle"))]
response = llm.invoke(messages)

In [ ]:
print(response.content)

`LangChain` workflow 2

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

text = "What would be a good name for a new national park with a {terrain} terrain."

template = PromptTemplate.from_template(text)
print(template.invoke({"terrain":"jungle"}))

llm_chain = template | llm
response = llm_chain.invoke({"terrain": "jungle"})
print(response.content)

In [ ]:
terrains = ["jungle", "desert", "coastal"]

for terrain in terrains:
    # create a prompt
    prompt_text = f"What would be a good name for a new national park with a {terrain} terrain."
    print(f"\n{prompt_text}")

    # create a HumanMessage - I think this step is not necessary
    messages =[HumanMessage(content=prompt_text)]

    # Pass message to model
    response = llm.invoke(messages)
    print(response.content)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key)

chat_template = ChatPromptTemplate.from_messages([
    ("system", "If given name of the terrain and wild animals, you will gnerate the name of the national park."),
    ("human" , "{text}")
])

# method 1
response = llm.invoke(chat_template.format_messages(text="bears on mountain"))
print(response.content)

# method 2
llm_chain = chat_template | llm
response = llm_chain.invoke(input=[{"text": "bears on mountains"}])
print(response.content)

In [ ]:
from langchain.prompts import ChatPromptTemplate

park_template = ChatPromptTemplate.from_messages([
    ("system", "You are an avid traveller of nature. You like to give suggestions of national parks for travel."),
    ("system", "You always make up a new name of a national park and suggest it."),
    ("human", "Hello, how are you doing?"),
    ("ai", "I'm doing well, thanks. Give me suggestions of animal or terrain."),
    ("human","{animal} {terrain}"),
    ("human", "{text}")
])

messages = park_template.format_messages(animal="bears", terrain="mountains", text="")
response = llm.invoke(messages)
print(response.content)

In [ ]:
messages = park_template.format_messages(
    animal="penguin",
    terrain="desert",
    text="")

response = llm(messages)
print(response.content)

In [ ]:
messages = park_template.format_messages(
    animal="",
    terrain="",
    text="I rather go to play video games, any suggestions")

response = llm(messages)
print(response.content)

### Examples of Chain

In [ ]:
# Import required libraries
import os  
from dotenv import load_dotenv  
from operator import itemgetter
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableLambda, RunnablePassthrough
from langchain.vectorstores import FAISS